# GIS — Distance to the MV grid and distance bands

**Study area:** 21 municipalities of Norte Amazónica (clusters C1–C5)
**Sources:**
- `comunidades_electricidad_2012.json` — community census points (electricity, 2012)
- `Centrales_Generadoras.json` — generation plants 2023
- `Líneas_de_Media_Tensión_2023.json` — MV lines 2023

**Tasks:**
1. Distance of unelectrified communities to the nearest MV line
2. Scale the 2012 spatial distribution to 2024 unelectrified HH (Source C)
3. Aggregate by cluster × distance band
4. Generation inventory

**CRS for distance computation:** EPSG:32719 (UTM Zone 19S) — appropriate for northern Bolivia (~-69° to -64° longitude)

In [1]:
import os
import json
import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path
from shapely.ops import nearest_points
import warnings
warnings.filterwarnings("ignore")

In [2]:
BASE_DIR   = Path(".")
DATA_DIR   = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

SOURCE_C_PATH = Path("../exctraction of data/output/municipalities_counts.csv")

CLUSTER_MAP = {
    "Exaltación":             "C1",
    "Ixiamas":                "C1",
    "Reyes":                  "C1",
    "Santa_Rosa_Beni":        "C1",
    "Bolpebra":               "C2",
    "Guayaramerín":           "C3",
    "Puerto_Gonzalo_Moreno":  "C3",
    "Riberalta":              "C3",
    "Bella_Flor":             "C4",
    "Filadelfia":             "C4",
    "Ingavi":                 "C4",
    "Nueva_Esperanza":        "C4",
    "Porvenir":               "C4",
    "Puerto_Rico":            "C4",
    "San_Lorenzo":            "C4",
    "San_Pedro":              "C4",
    "Santa_Rosa_Pando":       "C4",
    "Santos_Mercado":         "C4",
    "Sena":                   "C4",
    "Villa_Nueva":            "C4",
    "Cobija":                 "C5",
}

BAND_BINS   = [0, 5, 10, 25, 50, np.inf]
BAND_LABELS = ["<5 km", "5-10 km", "10-25 km", "25-50 km", ">50 km"]

UTM_CRS = "EPSG:32719"  # UTM Zone 19S, central meridian -69°

# Region bounding box (with buffer) used to select generation plants
REGION_BBOX = dict(min_lon=-70.0, max_lon=-63.5, min_lat=-13.0, max_lat=-9.5)

In [3]:
com_gdf = gpd.read_file(DATA_DIR / "comunidades_electricidad_2012.json")
com_gdf.set_crs("EPSG:4326", inplace=True, allow_override=True)
print(f"{len(com_gdf)} community features")
print(f"Columns: {list(com_gdf.columns)}")

gen_gdf = gpd.read_file(DATA_DIR / "Centrales_Generadoras.json")
gen_gdf.set_crs("EPSG:4326", inplace=True, allow_override=True)
print(f"{len(gen_gdf)} generation plants")

lineas_gdf = gpd.read_file(DATA_DIR / "Líneas_de_Media_Tensión_2023.json")
lineas_gdf.set_crs("EPSG:4326", inplace=True, allow_override=True)

source_c = pd.read_csv(SOURCE_C_PATH)
source_c = source_c[source_c["municipality"] != "TOTAL"].copy()
print(f"\nSource C: {len(source_c)} municipalities, total = {source_c['non_elec_hh'].sum()} unelec. HH")
print(source_c[["municipality", "non_elec_hh"]].to_string(index=False))

19279 community features
Columns: ['id', 'fid', 'OBJECTID', 'gid', 'codine', 'poblacion', 'vivpartic', 'Depto', 'Provincia', 'Municipio', 'Comunidad', 'Area', 'Hogares', 'Red_elec', 'Motor', 'Panel', 'Otra', 'Hog_Elec', 'No_tiene', 'Cobertura', 'geometry']
48 generation plants



Source C: 21 municipalities, total = 11389 unelec. HH
         municipality  non_elec_hh
           Exaltación          273
         Guayaramerín          825
                Reyes          711
            Riberalta         2925
      Santa_Rosa_Beni          407
              Ixiamas         1043
           Bella_Flor          351
             Bolpebra          253
               Cobija          395
           Filadelfia          594
               Ingavi          261
      Nueva_Esperanza          151
             Porvenir          213
Puerto_Gonzalo_Moreno          348
          Puerto_Rico          325
          San_Lorenzo          302
            San_Pedro          369
     Santa_Rosa_Pando          149
       Santos_Mercado          208
                 Sena         1073
          Villa_Nueva          213


In [4]:
# A former export of the Líneas file was a copy of Centrales_Generadoras (48 Point features).
# Stop if that happens again: no fallback on the plant proxy.
_geom_types = lineas_gdf.geometry.geom_type.unique()
print(f"Líneas feature count : {len(lineas_gdf)}")
print(f"Líneas geometry types: {_geom_types}")
print(f"Líneas columns       : {sorted(set(lineas_gdf.columns) - {'geometry'})}")

print("\nSIN field — feature count per value:")
if "SIN" in lineas_gdf.columns:
    print(lineas_gdf["SIN"].value_counts().to_string())
else:
    print("  SIN field NOT FOUND")

with open(DATA_DIR / "Centrales_Generadoras.json", "rb") as _f:
    _gen_bytes = _f.read()
with open(DATA_DIR / "Líneas_de_Media_Tensión_2023.json", "rb") as _f:
    _ln_bytes = _f.read()

_v_geom  = set(_geom_types).issubset({"LineString", "MultiLineString"})
_v_cnt   = len(lineas_gdf) != 48
_v_sin   = "SIN" in lineas_gdf.columns
_v_bytes = _ln_bytes != _gen_bytes
if not (_v_geom and _v_cnt and _v_sin and _v_bytes):
    raise RuntimeError("Líneas_de_Media_Tensión_2023.json is not an MV line layer.")

Líneas feature count : 653
Líneas geometry types: ['MultiLineString']
Líneas columns       : ['OBJECTID', 'SHAPE_Leng', 'SIN', 'Titular', 'fid', 'id']

SIN field — feature count per value:
SIN
SIN           583
Sist. Ais.     70


In [5]:
# Communities of the 21 study municipalities: (standard_name, dept_substring, muni_substring),
# matched case-insensitively on Depto and Municipio.
MUNI_FILTER = [
    ("Exaltación",            "Beni",   "Exaltac"),
    ("Guayaramerín",          "Beni",   "Guayaramer"),    # encoding-safe prefix
    ("Reyes",                 "Beni",   "Reyes"),
    ("Riberalta",             "Beni",   "Riberalta"),
    ("Santa_Rosa_Beni",       "Beni",   "Santa Rosa"),    # 'Tercera Sección - Santa Rosa'
    ("Ixiamas",               "La Paz", "Ixiamas"),
    ("Bella_Flor",            "Pando",  "Bella Flor"),
    ("Bolpebra",              "Pando",  "Bolpebra"),
    ("Cobija",                "Pando",  "Cobija"),
    ("Filadelfia",            "Pando",  "Filadelfia"),
    ("Ingavi",                "Pando",  "Ingavi"),
    ("Nueva_Esperanza",       "Pando",  "Nueva Esperanza"),
    ("Porvenir",              "Pando",  "Porvenir"),
    ("Puerto_Gonzalo_Moreno", "Pando",  "Puerto Gonzalo"),
    ("Puerto_Rico",           "Pando",  "Puerto Rico"),
    ("San_Lorenzo",           "Pando",  "San Lorenzo"),
    ("San_Pedro",             "Pando",  "San Pedro"),
    ("Santa_Rosa_Pando",      "Pando",  "Santa Rosa del Abun"),  # 'Santa Rosa del Abuná'
    ("Santos_Mercado",        "Pando",  "Santos Mercado"),
    ("Sena",                  "Pando",  "Sena"),
    ("Villa_Nueva",           "Pando",  "Villa Nueva"),
]

parts = []
for std_name, dept, muni_sub in MUNI_FILTER:
    mask = (
        com_gdf["Depto"].str.contains(dept, case=False, na=False) &
        com_gdf["Municipio"].str.contains(muni_sub, case=False, na=False)
    )
    sub = com_gdf[mask].copy()
    sub["municipio_std"] = std_name
    sub["cluster"]       = CLUSTER_MAP[std_name]
    parts.append(sub)

region_gdf = pd.concat(parts, ignore_index=True)
region_gdf["No_tiene"]  = pd.to_numeric(region_gdf["No_tiene"],  errors="coerce").fillna(0)
region_gdf["poblacion"] = pd.to_numeric(region_gdf["poblacion"], errors="coerce").fillna(0)
region_gdf["Hogares"]   = pd.to_numeric(region_gdf["Hogares"],   errors="coerce").fillna(0)

print(f"Study area: {len(region_gdf)} communities across {region_gdf['municipio_std'].nunique()} municipalities")
print(f"Communities with No_tiene > 0: {(region_gdf['No_tiene'] > 0).sum()}")

summary_region = region_gdf.groupby("municipio_std").agg(
    N_communities=("municipio_std", "count"),
    Sum_No_tiene=("No_tiene", "sum"),
    Sum_Hogares=("Hogares", "sum"),
    cluster=("cluster", "first"),
).sort_values("cluster")
print("\n" + summary_region.to_string())

Study area: 727 communities across 21 municipalities
Communities with No_tiene > 0: 697

                       N_communities  Sum_No_tiene  Sum_Hogares cluster
municipio_std                                                          
Exaltación                        24         271.0        938.0      C1
Santa_Rosa_Beni                   29         311.0       1976.0      C1
Ixiamas                           79         986.0       2270.0      C1
Reyes                             48         963.0       3018.0      C1
Bolpebra                          16         306.0        548.0      C2
Puerto_Gonzalo_Moreno             21         370.0       1026.0      C3
Guayaramerín                      49         961.0       9260.0      C3
Riberalta                         94        2562.0      19588.0      C3
Santos_Mercado                    15         200.0        394.0      C4
Santa_Rosa_Pando                  15         220.0        652.0      C4
San_Pedro                         17         42

In [6]:
# Task 1 — distance to the nearest MV line, in EPSG:32719 (metric)
region_utm = region_gdf.to_crs(UTM_CRS)
lineas_utm = lineas_gdf.to_crs(UTM_CRS)

unelec = region_utm[region_utm["No_tiene"] > 0].copy()
print(f"Unelectrified communities (No_tiene > 0): {len(unelec)}")

ref_union = lineas_utm.geometry.unary_union
unelec["distance_km"] = unelec.geometry.apply(
    lambda geom: geom.distance(nearest_points(geom, ref_union)[1]) / 1000)

unelec["dist_band"] = pd.cut(
    unelec["distance_km"],
    bins=BAND_BINS,
    labels=BAND_LABELS,
    right=True,
    include_lowest=True,
)

print(f"Distance range: {unelec['distance_km'].min():.1f} – {unelec['distance_km'].max():.1f} km")
print(f"Median distance: {unelec['distance_km'].median():.1f} km")

print("\nUnelectrified communities by distance band (2012 No_tiene HH):")
band_summary = unelec.groupby("dist_band", observed=True).agg(
    n_communities=("municipio_std", "count"),
    hh_no_tiene=("No_tiene", "sum"),
).assign(share=lambda df: (df["hh_no_tiene"] / df["hh_no_tiene"].sum()).round(3))
print(band_summary.to_string())

Unelectrified communities (No_tiene > 0): 697


Distance range: 0.0 – 128.1 km
Median distance: 12.6 km

Unelectrified communities by distance band (2012 No_tiene HH):
           n_communities  hh_no_tiene  share
dist_band                                   
<5 km                248       6251.0  0.476
5-10 km               67       1058.0  0.081
10-25 km             139       1933.0  0.147
25-50 km             111       1794.0  0.137
>50 km               132       2096.0  0.160


In [7]:
# Task 2 — scale 2012 geography to 2024 quantities (Source C).
# Assumption: the 2012 distribution of No_tiene across communities still holds in 2024:
#   hh_2024_i = HH_2024_muni × (No_tiene_2012_i / Σ No_tiene_2012_in_muni)
source_c_dict = source_c.set_index("municipality")["non_elec_hh"].to_dict()
unelec["total_2024"] = unelec["municipio_std"].map(source_c_dict)

missing = unelec[unelec["total_2024"].isna()]["municipio_std"].unique()
if len(missing) > 0:
    print(f"WARNING: No Source C match found for: {missing}")
    print("  Check that municipality names match between CLUSTER_MAP and municipalities_counts.csv")

muni_sum_2012 = unelec.groupby("municipio_std")["No_tiene"].transform("sum")
unelec["weight_2012"]    = unelec["No_tiene"] / muni_sum_2012.replace(0, np.nan)
unelec["hh_2024_scaled"] = unelec["weight_2012"] * unelec["total_2024"]

check = unelec.groupby("municipio_std").agg(
    scaled_sum=("hh_2024_scaled", "sum"),
    source_c=("total_2024", "first"),
).round(1)
check["diff"] = (check["scaled_sum"] - check["source_c"]).abs()
print("Scaling validation (diff should be ~0):")
print(check.to_string())

total_scaled = unelec["hh_2024_scaled"].sum()
total_source_c = source_c["non_elec_hh"].sum()
print(f"\nTotal 2024 HH (scaled): {total_scaled:.1f}")
print(f"Total Source C:          {total_source_c}")

Scaling validation (diff should be ~0):
                       scaled_sum  source_c  diff
municipio_std                                    
Bella_Flor                  351.0       351   0.0
Bolpebra                    253.0       253   0.0
Cobija                      395.0       395   0.0
Exaltación                  273.0       273   0.0
Filadelfia                  594.0       594   0.0
Guayaramerín                825.0       825   0.0
Ingavi                      261.0       261   0.0
Ixiamas                    1043.0      1043   0.0
Nueva_Esperanza             151.0       151   0.0
Porvenir                    213.0       213   0.0
Puerto_Gonzalo_Moreno       348.0       348   0.0
Puerto_Rico                 325.0       325   0.0
Reyes                       711.0       711   0.0
Riberalta                  2925.0      2925   0.0
San_Lorenzo                 302.0       302   0.0
San_Pedro                   369.0       369   0.0
Santa_Rosa_Beni             407.0       407   0.0
Santa_Rosa

In [8]:
# Task 3 — aggregate by cluster × distance band
task3 = unelec.groupby(["cluster", "dist_band"], observed=True).agg(
    HH_2024_scaled   = ("hh_2024_scaled",  "sum"),
    N_communities    = ("municipio_std",    "count"),
    Mean_dist_km     = ("distance_km",      "mean"),
    Median_dist_km   = ("distance_km",      "median"),
).reset_index()

cluster_totals = unelec.groupby("cluster")["hh_2024_scaled"].sum().rename("cluster_total_2024")
task3 = task3.merge(cluster_totals, on="cluster")
task3["Share_of_cluster"] = (task3["HH_2024_scaled"] / task3["cluster_total_2024"]).round(4)

task3["HH_2024_scaled"]  = task3["HH_2024_scaled"].round(1)
task3["Mean_dist_km"]    = task3["Mean_dist_km"].round(2)
task3["Median_dist_km"]  = task3["Median_dist_km"].round(2)

task3 = task3.rename(columns={"dist_band": "Dist_band", "cluster": "Cluster"})
task3 = task3[["Cluster", "Dist_band", "HH_2024_scaled", "N_communities",
               "Mean_dist_km", "Median_dist_km", "Share_of_cluster"]]
print(task3.to_string(index=False))

print("\nShare of unelectrified HH beyond 50 km per cluster:")
beyond50 = task3[task3["Dist_band"] == ">50 km"].set_index("Cluster")["Share_of_cluster"]
for cl in sorted(CLUSTER_MAP.values()):
    sh = beyond50.get(cl, 0.0)
    print(f"  {cl}: {sh*100:.1f}%")

task3.to_csv(OUTPUT_DIR / "task3_cluster_band.csv", index=False, encoding="utf-8-sig")

Cluster Dist_band  HH_2024_scaled  N_communities  Mean_dist_km  Median_dist_km  Share_of_cluster
     C1     <5 km           718.9             42          0.53            0.11            0.2954
     C1   5-10 km           149.2             12          6.95            6.73            0.0613
     C1  10-25 km           227.5             19         16.85           16.27            0.0934
     C1  25-50 km           282.8             21         37.10           39.02            0.1162
     C1    >50 km          1055.6             70         85.91           86.80            0.4337
     C2     <5 km            45.5              2          0.09            0.09            0.1797
     C2   5-10 km            15.7              1          8.06            8.06            0.0621
     C2  10-25 km            95.9              7         19.78           20.60            0.3791
     C2  25-50 km            40.5              3         42.43           42.19            0.1601
     C2    >50 km            5

In [9]:
cluster_summary = unelec.groupby("cluster").agg(
    HH_2024_total    = ("hh_2024_scaled", "sum"),
    N_communities    = ("municipio_std",   "count"),
    Mean_dist_km     = ("distance_km",     "mean"),
    Median_dist_km   = ("distance_km",     "median"),
    Max_dist_km      = ("distance_km",     "max"),
).round(2)
print(cluster_summary.to_string())

         HH_2024_total  N_communities  Mean_dist_km  Median_dist_km  Max_dist_km
cluster                                                                         
C1              2434.0            164         44.01           38.94       128.11
C2               253.0             16         27.87           22.57        62.44
C3              4098.0            160         19.39            8.42       114.84
C4              4209.0            343         18.57           11.90        79.03
C5               395.0             14          0.59            0.03         3.54


In [10]:
# Task 4 — generation inventory of the study region
bbox = REGION_BBOX
gen_region = gen_gdf[
    (gen_gdf.geometry.x >= bbox["min_lon"]) &
    (gen_gdf.geometry.x <= bbox["max_lon"]) &
    (gen_gdf.geometry.y >= bbox["min_lat"]) &
    (gen_gdf.geometry.y <= bbox["max_lat"])
].copy()

print(f"Generation plants in study region ({len(gen_region)} total):")
print(gen_region[["Central", "Tipo", "Pot_kW", "Propietari"]].to_string(index=False))

print("\nInventory by technology type:")
inventory = gen_region.groupby("Tipo").agg(
    N_plants     = ("Central", "count"),
    Total_kW     = ("Pot_kW",  "sum"),
).sort_values("Total_kW", ascending=False)
inventory["Share_%"] = (inventory["Total_kW"] / inventory["Total_kW"].sum() * 100).round(1)
print(inventory.to_string())
print(f"\nTotal installed capacity in region: {gen_region['Pot_kW'].sum():.0f} kW")

hydro = gen_region[gen_region["Tipo"].str.contains("[Hh]idro|[Hh]ydro", na=False, regex=True)]
if len(hydro) > 0:
    print(f"\nHydro plants found ({len(hydro)}): EnergyScope should include a hydro technology.")
    print(hydro[["Central", "Tipo", "Pot_kW", "Propietari"]].to_string(index=False))
else:
    print("\nNo hydro plants in study region: no hydro generation in EnergyScope for Norte Amazónica.")

gen_region[["Central", "Tipo", "Pot_kW", "Propietari"]].sort_values(
    ["Tipo", "Pot_kW"], ascending=[True, False]
).to_csv(OUTPUT_DIR / "task4_generation_inventory.csv", index=False, encoding="utf-8-sig")

Generation plants in study region (22 total):
                   Central                            Tipo  Pot_kW                Propietari
        Cachuela Esperanza Termoelectrica Diesel o Gas Oil   840.0             ENDE DELEBENI
                 Riberalta Termoelectrica Diesel o Gas Oil 31400.0                      ENDE
                  La Salud Termoelectrica Diesel o Gas Oil    80.0              ENDE DELBENI
                    Calama Termoelectrica Diesel o Gas Oil    24.0              ENDE DELBENI
                    Cayoba Termoelectrica Diesel o Gas Oil   180.0              ENDE DELBENI
          Rosario del Yata Termoelectrica Diesel o Gas Oil  1210.0              ENDE DELBENI
       Planta Solar Cobija               Solar Fotovotaica  5100.0           ENDE Guaracachi
              Puerto Siles Termoelectrica Diesel o Gas Oil    10.0              ENDE DELBENI
               Cooperativa Termoelectrica Diesel o Gas Oil    16.0              ENDE DELBENI
San Juan de Puerto Ustar

In [11]:
print(f"Communities in study area:     {len(region_gdf):>6}")
print(f"Unelectrified communities:     {len(unelec):>6}  (No_tiene > 0)")
print(f"Total 2024 unelec. HH (src C): {int(source_c['non_elec_hh'].sum()):>6}")
print(f"Total 2024 HH (scaled check):  {unelec['hh_2024_scaled'].sum():>9.1f}")

Communities in study area:        727
Unelectrified communities:        697  (No_tiene > 0)
Total 2024 unelec. HH (src C):  11389
Total 2024 HH (scaled check):    11389.0


## Notes

### Assumption — Task 2 scaling
The 2012 community-level distribution of `No_tiene` (unelectrified households from the INE census) is assumed to remain representative in 2024. Each community receives a proportional share of its municipality's 2024 Source C total.

### CRS
EPSG:32719 (UTM Zone 19S, central meridian −69°) was chosen as it centres the study area (−68° to −64° longitude). Maximum scale distortion at the eastern edge (~−64°) is <0.1%.

### Distances by line type
The cells below split the distance by line type (SIN vs isolated systems) and compare the line
distances with the earlier generation-plant proxy (`output/task1_community_distances.csv`).

In [12]:
# Task 1 by line type. Four columns per community:
#   distance_km            — distance to nearest MV line (any SIN type)
#   nearest_line_type      — SIN value of that nearest line
#   distance_SIN_km        — distance to nearest SIN (interconnected) line
#   distance_isolated_km   — distance to nearest Sist. Ais. (isolated) line
lineas_utm2     = lineas_gdf.to_crs(UTM_CRS)
lineas_SIN      = lineas_utm2[lineas_utm2["SIN"] == "SIN"]
lineas_isolated = lineas_utm2[lineas_utm2["SIN"] == "Sist. Ais."]
print(f"SIN lines      : {len(lineas_SIN)}")
print(f"Isolated lines : {len(lineas_isolated)}")

union_all_l = lineas_utm2.geometry.unary_union
union_SIN_l = lineas_SIN.geometry.unary_union
union_iso_l = lineas_isolated.geometry.unary_union

# Reuses unelec (697 communities, hh_2024_scaled from Task 2) without its distance columns.
unelec_lines = unelec.drop(
    columns=["distance_km", "dist_band", "weight_2012", "total_2024"], errors="ignore"
).copy()
assert len(unelec_lines) == 697, f"Expected 697 communities, got {len(unelec_lines)}"

import shapely as _shp
_pts = np.array(unelec_lines.geometry)

unelec_lines["distance_km"]          = _shp.distance(_pts, union_all_l) / 1000
unelec_lines["distance_SIN_km"]      = _shp.distance(_pts, union_SIN_l) / 1000
unelec_lines["distance_isolated_km"] = _shp.distance(_pts, union_iso_l) / 1000
unelec_lines["nearest_line_type"] = np.where(
    unelec_lines["distance_SIN_km"] <= unelec_lines["distance_isolated_km"],
    "SIN", "Sist. Ais."
)

unelec_lines["dist_band"] = pd.cut(
    unelec_lines["distance_km"],
    bins=BAND_BINS, labels=BAND_LABELS, right=True, include_lowest=True,
)

print(f"\ndistance_km (nearest any MV line):")
print(f"  Range  : {unelec_lines['distance_km'].min():.2f} – {unelec_lines['distance_km'].max():.2f} km")
print(f"  Median : {unelec_lines['distance_km'].median():.2f} km")
print(f"  Mean   : {unelec_lines['distance_km'].mean():.2f} km")
print(f"\ndistance_SIN_km      : {unelec_lines['distance_SIN_km'].min():.2f}–{unelec_lines['distance_SIN_km'].max():.2f} km")
print(f"distance_isolated_km : {unelec_lines['distance_isolated_km'].min():.2f}–{unelec_lines['distance_isolated_km'].max():.2f} km")

print(f"\nNearest line type counts:")
print(unelec_lines["nearest_line_type"].value_counts().to_string())

print("\nBand distribution (No_tiene 2012 weighted):")
_bs = unelec_lines.groupby("dist_band", observed=True).agg(
    n_communities=("municipio_std", "count"),
    hh_no_tiene=("No_tiene", "sum"),
).assign(share=lambda d: (d["hh_no_tiene"] / d["hh_no_tiene"].sum()).round(3))
print(_bs.to_string())

SIN lines      : 583
Isolated lines : 70



distance_km (nearest any MV line):
  Range  : 0.00 – 128.11 km
  Median : 12.60 km
  Mean   : 24.60 km

distance_SIN_km      : 0.00–398.04 km
distance_isolated_km : 0.00–253.01 km

Nearest line type counts:
nearest_line_type
Sist. Ais.    584
SIN           113

Band distribution (No_tiene 2012 weighted):
           n_communities  hh_no_tiene  share
dist_band                                   
<5 km                248       6251.0  0.476
5-10 km               67       1058.0  0.081
10-25 km             139       1933.0  0.147
25-50 km             111       1794.0  0.137
>50 km               132       2096.0  0.160


In [13]:
# Communities inside Riberalta and Cobija must be ~0 km from the MV grid running through them;
# otherwise the CRS or the geometry is wrong.

for muni, threshold_km in [("Riberalta", 2.0), ("Cobija", 2.0)]:
    sub = unelec_lines[unelec_lines["municipio_std"] == muni].sort_values("distance_km")
    cols = ["Comunidad", "distance_km", "distance_SIN_km", "distance_isolated_km", "nearest_line_type"]
    print(f"── {muni}  ({len(sub)} unelectrified communities, sorted by distance_km) ──")
    print(sub[cols].head(5).to_string(index=False))
    min_d = sub["distance_km"].min()
    if min_d > threshold_km:
        print(f"  !!! WARNING: minimum = {min_d:.2f} km  (expected < {threshold_km} km)")
        print(f"       Check CRS alignment between community points and MV line layer.")
    else:
        print(f"  minimum distance = {min_d:.3f} km")
    print()

── Riberalta  (92 unelectrified communities, sorted by distance_km) ──
   Comunidad  distance_km  distance_SIN_km  distance_isolated_km nearest_line_type
     PALMIRA     0.000047       258.307271              0.000047        Sist. Ais.
   RIBERALTA     0.001745       269.008341              0.001745        Sist. Ais.
 SANTA MARIA     0.002410       252.276289              0.002410        Sist. Ais.
LA ESPERANZA     0.004578       263.549265              0.004578        Sist. Ais.
VILLA COTOCA     0.006873       264.351091              0.006873        Sist. Ais.
  minimum distance = 0.000 km

── Cobija  (14 unelectrified communities, sorted by distance_km) ──
    Comunidad  distance_km  distance_SIN_km  distance_isolated_km nearest_line_type
  VILLA BUSCH     0.006102       303.663587              0.006102        Sist. Ais.
   MEJILLONES     0.006109       306.331512              0.006109        Sist. Ais.
VILLA ROSARIO     0.007653       298.893630              0.007653        Sist. A

In [14]:
# Full per-community distance table, with the plant-proxy distance of the earlier run alongside.
_plant_csv = OUTPUT_DIR / "task1_community_distances.csv"
if _plant_csv.exists():
    _old = pd.read_csv(_plant_csv, encoding="utf-8-sig")
    _plant = _old[["Municipio_std", "Comunidad", "distance_km"]].rename(
        columns={"distance_km": "distance_plant_km", "Municipio_std": "municipio_std"}
    )
    # Some community names repeat within a municipality.
    _plant = _plant.drop_duplicates(subset=["municipio_std", "Comunidad"], keep="first")
    _working = unelec_lines.merge(_plant, on=["municipio_std", "Comunidad"], how="left")
    n_matched = _working["distance_plant_km"].notna().sum()
    print(f"Matched {n_matched}/{len(_working)} communities to plant-proxy distances from old CSV.")
    if n_matched < len(_working):
        print(f"  Note: {len(_working) - n_matched} communities unmatched.")
else:
    print(f"WARNING: {_plant_csv} not found — distance_plant_km will be absent.")
    _working = unelec_lines.copy()
    _working["distance_plant_km"] = float("nan")

task1_lines_out = _working[[
    "municipio_std", "cluster", "Municipio", "Comunidad",
    "No_tiene", "poblacion", "hh_2024_scaled",
    "distance_km", "nearest_line_type",
    "distance_SIN_km", "distance_isolated_km",
    "distance_plant_km", "dist_band",
]].copy()
task1_lines_out.columns = [
    "Municipio_std", "Cluster", "Municipio_raw", "Comunidad",
    "No_tiene_2012", "Poblacion_2012", "HH_2024_scaled",
    "distance_km", "nearest_line_type",
    "distance_SIN_km", "distance_isolated_km",
    "distance_plant_km", "Dist_band_lines",
]
task1_lines_out = task1_lines_out.sort_values(
    ["Cluster", "Municipio_std", "distance_km"]
).reset_index(drop=True)

task1_lines_out.to_csv(OUTPUT_DIR / "task1_community_distances_lines.csv", index=False, encoding="utf-8-sig")
print(f"\n{task1_lines_out.shape[0]} rows × {task1_lines_out.shape[1]} cols. Sample:")
print(task1_lines_out[[
    "Municipio_std", "Cluster", "Comunidad",
    "distance_km", "nearest_line_type", "distance_SIN_km",
    "distance_isolated_km", "distance_plant_km",
]].head(5).to_string(index=False))

Matched 697/697 communities to plant-proxy distances from old CSV.

697 rows × 13 cols. Sample:
Municipio_std Cluster       Comunidad  distance_km nearest_line_type  distance_SIN_km  distance_isolated_km  distance_plant_km
   Exaltación      C1      EXALTACION     0.046616               SIN         0.046616              0.055350          48.678780
   Exaltación      C1       BOCORONDO     6.621175               SIN         6.621175              8.147679          42.851054
   Exaltación      C1     20 DE ENERO     9.233733               SIN         9.233733             14.086965          62.050105
   Exaltación      C1       TROMPILLO    25.196976               SIN        25.196976             26.658686          36.711807
   Exaltación      C1 PUERTO SANTIAGO    26.138378               SIN        26.138378             27.623388          35.615363


In [15]:
# Band shift from the plant proxy to real MV line distances, weighted by hh_2024_scaled.
#   share_plant = HH share with the plant proxy distance
#   share_lines = HH share with the MV line distance
#   Δ_pp        = (share_lines − share_plant) × 100
_plant_csv = OUTPUT_DIR / "task1_community_distances.csv"
if not _plant_csv.exists():
    raise FileNotFoundError(
        f"{_plant_csv} not found. Run the original notebook first to generate plant-proxy distances."
    )

_old_df = pd.read_csv(_plant_csv, encoding="utf-8-sig")
_old_df = _old_df.rename(columns={
    "Municipio_std": "municipio_std",
    "distance_km":   "distance_plant_km",
})
_old_df_dedup = _old_df[["municipio_std","Comunidad","distance_plant_km"]].drop_duplicates(
    subset=["municipio_std","Comunidad"], keep="first"
)

_diff_base = unelec_lines.merge(_old_df_dedup, on=["municipio_std","Comunidad"], how="left")
n_merged = _diff_base["distance_plant_km"].notna().sum()
print(f"Merged plant distances: {n_merged}/{len(_diff_base)} communities matched.")
_diff_base = _diff_base.dropna(subset=["distance_plant_km","hh_2024_scaled"])

_diff_base["dist_band_plant"] = pd.cut(
    _diff_base["distance_plant_km"],
    bins=BAND_BINS, labels=BAND_LABELS, right=True, include_lowest=True,
)

def _cluster_band_shares(df, band_col):
    grp = df.groupby(["cluster", band_col], observed=True)["hh_2024_scaled"].sum().reset_index()
    grp.columns = ["cluster", "dist_band", "hh"]
    tot = df.groupby("cluster")["hh_2024_scaled"].sum().rename("cluster_total")
    grp = grp.merge(tot, on="cluster")
    grp["share"] = grp["hh"] / grp["cluster_total"]
    return grp

_plant_shares = _cluster_band_shares(_diff_base, "dist_band_plant").rename(
    columns={"hh": "hh_plant", "share": "share_plant"}
)
_line_shares = _cluster_band_shares(_diff_base, "dist_band").rename(
    columns={"hh": "hh_lines", "share": "share_lines"}
)

# Categorical -> str so that fillna(0) works after the outer merge.
_plant_shares["dist_band"] = _plant_shares["dist_band"].astype(str)
_line_shares["dist_band"]  = _line_shares["dist_band"].astype(str)

_diff = pd.merge(
    _plant_shares[["cluster","dist_band","hh_plant","share_plant"]],
    _line_shares [["cluster","dist_band","hh_lines","share_lines"]],
    on=["cluster","dist_band"],
    how="outer",
).fillna(0)

_band_order = {b: i for i, b in enumerate(BAND_LABELS)}
_diff["_ord"] = _diff["dist_band"].map(_band_order)
_diff = _diff.sort_values(["cluster","_ord"]).drop(columns=["_ord"])

_diff["Δ_pp"] = ((_diff["share_lines"] - _diff["share_plant"]) * 100).round(1)

_out = _diff.rename(columns={"cluster":"Cluster","dist_band":"Band"})
print(_out[["Cluster","Band","hh_plant","share_plant","hh_lines","share_lines","Δ_pp"]]
      .to_string(index=False, float_format=lambda x: f"{x:.3f}"))

print("\nChange in >50 km share per cluster (flagged if |Δ| > 5 pp):")
_b50 = _diff[_diff["dist_band"] == ">50 km"].set_index("cluster")
for cl in sorted(set(CLUSTER_MAP.values())):
    if cl in _b50.index:
        row = _b50.loc[cl]
        flag = "  *** FLAGGED" if abs(row["Δ_pp"]) > 5 else ""
        print(f"  {cl}:  plant={row['share_plant']*100:.1f}%  "
              f"lines={row['share_lines']*100:.1f}%  "
              f"Δ={row['Δ_pp']:+.1f} pp{flag}")
    else:
        print(f"  {cl}:  no communities in >50 km band")

Merged plant distances: 697/697 communities matched.
Cluster     Band  hh_plant  share_plant  hh_lines  share_lines    Δ_pp
     C1    <5 km    68.502        0.028   718.927        0.295  26.700
     C1  5-10 km    27.199        0.011   149.182        0.061   5.000
     C1 10-25 km    10.074        0.004   227.452        0.093   8.900
     C1 25-50 km   156.443        0.064   282.811        0.116   5.200
     C1   >50 km  2171.782        0.892  1055.629        0.434 -45.900
     C2    <5 km     0.000        0.000    45.474        0.180  18.000
     C2  5-10 km     0.000        0.000    15.709        0.062   6.200
     C2 10-25 km     0.000        0.000    95.908        0.379  37.900
     C2 25-50 km   130.634        0.516    40.513        0.160 -35.600
     C2   >50 km   122.366        0.484    55.395        0.219 -26.500
     C3    <5 km  1529.234        0.373  2504.541        0.611  23.800
     C3  5-10 km   559.063        0.136   409.113        0.100  -3.700
     C3 10-25 km  1218.1

In [16]:
# Task 3 with the line distances (same columns as task3_cluster_band.csv).
task3_lines = unelec_lines.groupby(["cluster", "dist_band"], observed=True).agg(
    HH_2024_scaled = ("hh_2024_scaled", "sum"),
    N_communities  = ("municipio_std",   "count"),
    Mean_dist_km   = ("distance_km",     "mean"),
    Median_dist_km = ("distance_km",     "median"),
).reset_index()

_cl_tot = unelec_lines.groupby("cluster")["hh_2024_scaled"].sum().rename("cluster_total_2024")
task3_lines = task3_lines.merge(_cl_tot, on="cluster")
task3_lines["Share_of_cluster"] = (task3_lines["HH_2024_scaled"] / task3_lines["cluster_total_2024"]).round(4)

task3_lines["HH_2024_scaled"] = task3_lines["HH_2024_scaled"].round(1)
task3_lines["Mean_dist_km"]   = task3_lines["Mean_dist_km"].round(2)
task3_lines["Median_dist_km"] = task3_lines["Median_dist_km"].round(2)

task3_lines = task3_lines.rename(columns={"dist_band": "Dist_band", "cluster": "Cluster"})
task3_lines = task3_lines[[
    "Cluster", "Dist_band", "HH_2024_scaled", "N_communities",
    "Mean_dist_km", "Median_dist_km", "Share_of_cluster",
]]
print(task3_lines.to_string(index=False))

print("\nShare of HH in >50 km band per cluster (line distances):")
_b50l = task3_lines[task3_lines["Dist_band"] == ">50 km"].set_index("Cluster")["Share_of_cluster"]
for cl in sorted(set(CLUSTER_MAP.values())):
    sh = _b50l.get(cl, 0.0)
    print(f"  {cl}: {sh * 100:.1f}%")

task3_lines.to_csv(OUTPUT_DIR / "task3_cluster_band_lines.csv", index=False, encoding="utf-8-sig")

Cluster Dist_band  HH_2024_scaled  N_communities  Mean_dist_km  Median_dist_km  Share_of_cluster
     C1     <5 km           718.9             42          0.53            0.11            0.2954
     C1   5-10 km           149.2             12          6.95            6.73            0.0613
     C1  10-25 km           227.5             19         16.85           16.27            0.0934
     C1  25-50 km           282.8             21         37.10           39.02            0.1162
     C1    >50 km          1055.6             70         85.91           86.80            0.4337
     C2     <5 km            45.5              2          0.09            0.09            0.1797
     C2   5-10 km            15.7              1          8.06            8.06            0.0621
     C2  10-25 km            95.9              7         19.78           20.60            0.3791
     C2  25-50 km            40.5              3         42.43           42.19            0.1601
     C2    >50 km            5